# SQLite Basics

### Schema Setup

Creating the SQLite database tables and checks the schema so the data model is ready before loading F1 information.

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("../data/SQLite_basics.db")

cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON")

cursor.executescript("""
    CREATE TABLE IF NOT EXISTS circuits (
        circuit_id      INTEGER PRIMARY KEY,
        circuit_name    TEXT    NOT NULL UNIQUE,
        circuit_country TEXT    NOT NULL,
        track_length_m  REAL    NOT NULL CHECK (track_length_m > 0)
    );

    CREATE TABLE IF NOT EXISTS drivers (
        driver_id   INTEGER PRIMARY KEY,
        driver_code TEXT    NOT NULL UNIQUE,
        team_name   TEXT    NOT NULL
    );

    CREATE TABLE IF NOT EXISTS sessions (
        session_id      INTEGER PRIMARY KEY,
        circuit_id      INTEGER NOT NULL REFERENCES circuits(circuit_id),
        session_date    TEXT    NOT NULL,
        session_type    TEXT    NOT NULL 
                        CHECK (session_type IN ('PRACTICE','QUALIFYING','RACE')),
        weather         TEXT
    );

    CREATE TABLE IF NOT EXISTS laps (
        lap_id      INTEGER PRIMARY KEY AUTOINCREMENT,
        session_id  INTEGER NOT NULL REFERENCES sessions(session_id),
        driver_id   INTEGER NOT NULL REFERENCES drivers(driver_id),
        lap_number  INTEGER NOT NULL,
        lap_time_s  REAL    CHECK (lap_time_s > 0 OR lap_time_s IS NULL),
        compound    TEXT    CHECK (compound IN 
                    ('SOFT','MEDIUM','HARD','INTER','WET')),
        speed_kmh   REAL    CHECK (speed_kmh > 0 OR speed_kmh IS NULL),
        is_pit_lap  INTEGER DEFAULT 0 CHECK (is_pit_lap IN (0,1)),
        is_valid    INTEGER DEFAULT 1 CHECK (is_valid IN (0,1)),
        UNIQUE(session_id, driver_id, lap_number)
    );
""")
conn.commit()
print("Schema created.")

result = pd.read_sql_query("""
    SELECT name FROM sqlite_master WHERE type='table'
""", conn)
print(result)

Schema created.
              name
0         circuits
1          drivers
2         sessions
3             laps
4  sqlite_sequence


### Load Track and Driver Data

Inserting the initial circuit, driver, session, and lap records needed for the SQLite examples.

In [2]:
cursor.executemany("INSERT OR IGNORE INTO circuits VALUES (?,?,?,?)", [
    (1, "Monaco",  "Monaco",  3.337),
    (2, "Hergla",  "Tunisia", 1.200),
])

cursor.executemany("INSERT OR IGNORE INTO drivers VALUES (?,?,?)", [
    (1, "VER", "Red Bull Racing"),
    (2, "HAM", "Mercedes"),
    (3, "LEC", "Ferrari"),
])

cursor.executemany("INSERT OR IGNORE INTO sessions VALUES (?,?,?,?,?)", [
    (1, 1, "2024-05-24", "PRACTICE","Dry"),
    (2, 1, "2024-05-25", "QUALIFYING", "Dry"),
    (3, 1, "2024-05-26", "RACE",  "Dry"),
    (4, 2, "2025-03-15", "PRACTICE","Dry"),
    (5, 2, "2025-03-17", "RACE",  "Dry"),
])
cursor.executemany("INSERT OR IGNORE INTO laps VALUES (?,?,?,?,?,?,?,?,?)", [
    (1, 1, 1, 1, 75.3, "SOFT", 180.3, 0, 1),
    (2, 1, 1, 2, 74.8, "SOFT", 181.0, 1, 0),
    (3, 1, 1, 3, 76.0, "HARD", 179.5, 0, 1),
    (4, 1, 2, 5, 76.5, "MEDIUM", 178.9, 1, 0),
    (5, 1, 2, 6, 75.9, "SOFT", 180.0, 0, 1),
    (6, 1, 2, 7, 77.0, "SOFT", 178.0, 1, 0),
    (7, 1, 3, 4, 77.5, "MEDIUM", 177.5, 0, 1),
    (8, 1, 3, 5, 76.8, "MEDIUM", 179.0, 1, 0),
    (9, 1, 3, 6, 78.0, "HARD", 176.5, 0, 1),
    (10, 2, 1, 1, 74.5, "MEDIUM", 182.0, 0, 1),
    (11, 2, 1, 2, 75.0, "MEDIUM", 183.0, 1, 0),
    (12, 2, 1, 3, 71.8, "SOFT", 184.0, 0, 1),
    (13, 2, 2, 1, 72.0, "SOFT", 181.5, 0, 1),
    (14, 2, 2, 2, 72.5, "SOFT", 182.0, 0, 1),
    (15, 2, 3, 1, 72.4, "SOFT", 179.0, 0, 1),
    (16, 2, 3, 2, 73.0, "SOFT", 180.0, 1, 0),
    (17, 3, 1, 1, 80.0, "HARD", 175.0, 0, 1),
    (18, 3, 1, 2, 79.4, "HARD", 176.0, 0, 1),
    (19, 3, 1, 3, 80.1, "HARD", 174.5, 0, 1),
    (20, 3, 1, 4, 80.0, "HARD", 173.5, 0, 1),
    (21, 3, 2, 1, 79.9, "HARD", 175.0, 0, 1),
    (22, 3, 2, 2, 79.5, "HARD", 176.0, 0, 1),
    (23, 3, 2, 3, 80.2, "HARD", 174.5, 0, 1),
    (24, 3, 2, 4, 80.3, "HARD", 173.5, 0, 1),
    (25, 3, 3, 1, 80.2, "HARD", 175.0, 0, 1),
    (26, 3, 3, 2, 79.7, "HARD", 176.0, 0, 1),
    (27, 3, 3, 3, 80.4, "HARD", 174.5, 0, 1),
    (28, 3, 3, 4, 80.3, "HARD", 173.5, 0, 1),
])
conn.commit()
result = pd.read_sql_query("SELECT * FROM circuits", conn)
print(result)

result = pd.read_sql_query("SELECT * FROM drivers", conn)
print(result)

result = pd.read_sql_query("SELECT * FROM sessions", conn)
print(result)

result = pd.read_sql_query("SELECT * FROM laps", conn)
print(result)

   circuit_id circuit_name circuit_country  track_length_m
0           1       Monaco          Monaco           3.337
1           2       Hergla         Tunisia           1.200
   driver_id driver_code        team_name
0          1         VER  Red Bull Racing
1          2         HAM         Mercedes
2          3         LEC          Ferrari
   session_id  circuit_id session_date session_type weather
0           1           1   2024-05-24     PRACTICE     Dry
1           2           1   2024-05-25   QUALIFYING     Dry
2           3           1   2024-05-26         RACE     Dry
3           4           2   2025-03-15     PRACTICE     Dry
4           5           2   2025-03-17         RACE     Dry
    lap_id  session_id  driver_id  lap_number  lap_time_s compound  speed_kmh  \
0        1           1          1           1        75.3     SOFT      180.3   
1        2           1          1           2        74.8     SOFT      181.0   
2        3           1          1           3       

## Query the Lap Data

Aggregating lap times, pit-stop data, and driver-session details to explore patterns in the racing data.

In [3]:
print(pd.read_sql_query("""
    SELECT d.driver_code, MIN(l.lap_time_s) AS fastest_lap_s
    FROM laps l
    JOIN drivers d ON l.driver_id = d.driver_id
    WHERE l.is_valid = 1
    GROUP BY d.driver_code
    ORDER BY fastest_lap_s ASC
""", conn))

print(pd.read_sql_query("""
    SELECT AVG(l.lap_time_s) as average_lap_time, l.compound
    FROM laps l
    WHERE l.is_valid = 1
    GROUP BY l.compound
""", conn))

print(pd.read_sql_query("""
    SELECT count(*) as pit_laps, d.driver_code , s.session_type
    FROM laps l
    JOIN drivers d ON l.driver_id = d.driver_id
    JOIN sessions s ON l.session_id = s.session_id
    WHERE l.is_pit_lap = 1
    GROUP BY d.driver_code, s.session_type
""", conn))

print(pd.read_sql_query("""
    SELECT d.driver_code, l.*, c.circuit_name
    FROM laps l
    JOIN drivers d ON l.driver_id = d.driver_id
    JOIN sessions s ON l.session_id = s.session_id
    JOIN circuits c ON s.circuit_id = c.circuit_id
    WHERE d.driver_code = 'VER'
""", conn))

  driver_code  fastest_lap_s
0         VER           71.8
1         HAM           72.0
2         LEC           72.4
   average_lap_time compound
0         79.571429     HARD
1         76.000000   MEDIUM
2         73.316667     SOFT
   pit_laps driver_code session_type
0         2         HAM     PRACTICE
1         1         LEC     PRACTICE
2         1         LEC   QUALIFYING
3         1         VER     PRACTICE
4         1         VER   QUALIFYING
  driver_code  lap_id  session_id  driver_id  lap_number  lap_time_s compound  \
0         VER       1           1          1           1        75.3     SOFT   
1         VER       2           1          1           2        74.8     SOFT   
2         VER       3           1          1           3        76.0     HARD   
3         VER      10           2          1           1        74.5   MEDIUM   
4         VER      11           2          1           2        75.0   MEDIUM   
5         VER      12           2          1           3    

## Check Foreign Key Rules

This example confirms whether SQLite rejects an insert that violates the relationship between laps and drivers.

In [4]:
try:
    cursor.execute("""
        INSERT INTO laps (session_id, driver_id, lap_number, lap_time_s, compound)
        VALUES (1, 99, 1, 90.0, 'SOFT')
    """)
    conn.commit()
    print("Insert succeeded — referential integrity NOT enforced")
except Exception as e:
    print(f"Insert rejected: {e}")

Insert rejected: FOREIGN KEY constraint failed


- Before using "PRAGMA foreign_keys = ON" the exception doesn't happen and the insertion succeeds which creates a flaw in the security of the Data.

## Review the Final Analysis

This final section creates a cleaned pandas DataFrame and saves it into SQLite for a simple downstream analysis workflow.

In [5]:
df_analysis = pd.DataFrame({
    "driver": ["VER", "VER", "HAM", "HAM", "HAM"],
    "lap_number": [1, 3, 1, 2, 3],
    "lap_time_s": [90.1, 91.2, 90.8, 89.9, 90.3],
    "compound": ["SOFT", "SOFT", "MEDIUM", "MEDIUM", "MEDIUM"],
    "speed_kmh": [287.3, 288.4, 290.1, 289.7, 293.2],
    "is_valid": [1, 1, 1, 1, 1]
})
df_analysis.to_sql("cleaned_laps", conn, if_exists="replace", index=False)

df_back = pd.read_sql_query("""
    SELECT * FROM cleaned_laps
    WHERE driver = 'VER'
    ORDER BY lap_number ASC
""", conn)

print(df_back)
conn.close()

  driver  lap_number  lap_time_s compound  speed_kmh  is_valid
0    VER           1        90.1     SOFT      287.3         1
1    VER           3        91.2     SOFT      288.4         1
